# Notebook 03: Cleaning and Missing Value Analysis

This notebook cleans the merged FIA tree-level dataset created in Notebook 02.

The aim is to prepare a valid unencoded dataset for later EDA, feature engineering and modelling. The target variable is individual-tree above-ground carbon storage (`CARBON_AG`). The notebook removes unusable records, checks missing values, validates key biological measurements and saves a cleaned tree-level dataset.

This notebook does not perform final encoding, scaling, modelling or train/test splitting.

## 1. Import libraries and project paths

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.paths import INTERIM_DIR, OUTPUTS_DIR, create_project_dirs
from src.fia_cleaning import (
    load_merged_dataset,
    summarise_missing_values,
    clean_fia_tree_data,
    summarise_key_variables,
    save_cleaning_outputs,
    convert_key_columns_to_numeric
)

create_project_dirs()

print("Project root:", PROJECT_ROOT)
print("Interim folder:", INTERIM_DIR)
print("Outputs folder:", OUTPUTS_DIR)

Project root: e:\BA_DV_Project\COMM074_FIA_Project
Interim folder: E:\BA_DV_Project\COMM074_FIA_Project\data\interim
Outputs folder: E:\BA_DV_Project\COMM074_FIA_Project\outputs


## 2. Check merged dataset file

In [2]:
merged_path = INTERIM_DIR / "merged_tree_plot_cond.parquet"

print("Merged file exists:", merged_path.exists())
print("Merged file path:", merged_path)

Merged file exists: True
Merged file path: E:\BA_DV_Project\COMM074_FIA_Project\data\interim\merged_tree_plot_cond.parquet


## 3. Load merged tree-level dataset

In [3]:
merged_df = load_merged_dataset()

print("Merged dataset shape:", merged_df.shape)
merged_df.head()

Merged dataset shape: (5796627, 55)


,CN,PLT_CN,PREV_TRE_CN,INVYR,STATECD,UNITCD,COUNTYCD,PLOT,SUBP,TREE,...,FLDTYPCD,OWNGRPCD,RESERVCD,SITECLCD,STDAGE,STDSZCD,BALIVE,ALSTK,GSSTK,has_cond_match
0,157825670010854,157531323010854,NaN,1982,1,5,111,90055,104,3,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
1,157825671010854,157531323010854,NaN,1982,1,5,111,90055,104,4,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
2,157825672010854,157531323010854,NaN,1982,1,5,111,90055,104,5,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
3,157825673010854,157531323010854,NaN,1982,1,5,111,90055,104,6,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
4,157825674010854,157531324010854,NaN,1982,1,5,111,90056,101,1,...,NaN,40.0,0.0,4.0,5.0,3.0,NaN,160.0,150.8,True


In [4]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5796627 entries, 0 to 5796626
Data columns (total 55 columns):
 #   Column          Dtype  
---  ------          -----  
 0   CN              int64  
 1   PLT_CN          int64  
 2   PREV_TRE_CN     str    
 3   INVYR           int64  
 4   STATECD         int64  
 5   UNITCD          int64  
 6   COUNTYCD        int64  
 7   PLOT            int64  
 8   SUBP            int64  
 9   TREE            int64  
 10  CONDID          int64  
 11  STATUSCD        int64  
 12  SPCD            int64  
 13  DIA             float64
 14  DIAHTCD         float64
 15  HT              str    
 16  HTCD            str    
 17  ACTUALHT        str    
 18  CR              float64
 19  CCLCD           float64
 20  CARBON_AG       float64
 21  CARBON_BG       float64
 22  DRYBIO_AG       float64
 23  DRYBIO_BG       float64
 24  VOLCFNET        float64
 25  VOLCSNET        float64
 26  VOLBFNET        float64
 27  TPA_UNADJ       float64
 28  PLOT_CN_KEY     int64  

In [5]:
merged_df = convert_key_columns_to_numeric(merged_df)

## 4. Validate target and key variables

In [6]:
target_col = "CARBON_AG"

key_columns = [
    "CN", "PLT_CN", "CONDID", "TREE",
    "STATUSCD", "SPCD",
    "DIA", "HT", "ACTUALHT", "CR",
    "CARBON_AG",
    "STATECD", "COUNTYCD", "INVYR",
    "FORTYPCD", "OWNGRPCD", "SITECLCD", "STDAGE", "STDSZCD"
]

existing_key_columns = [c for c in key_columns if c in merged_df.columns]
missing_key_columns = [c for c in key_columns if c not in merged_df.columns]

print("Existing key columns:")
print(existing_key_columns)

print("\nMissing key columns:")
print(missing_key_columns)

Existing key columns:
['CN', 'PLT_CN', 'CONDID', 'TREE', 'STATUSCD', 'SPCD', 'DIA', 'HT', 'ACTUALHT', 'CR', 'CARBON_AG', 'STATECD', 'COUNTYCD', 'INVYR', 'FORTYPCD', 'OWNGRPCD', 'SITECLCD', 'STDAGE', 'STDSZCD']

Missing key columns:
[]


## 5. Initial missing value summary

In [7]:
initial_missing_summary = summarise_missing_values(merged_df)

initial_missing_summary.head(30)

,variable,missing_count,missing_percentage,dtype
0,VOLBFNET,3992603,68.878039,float64
1,VOLCSNET,3992575,68.877556,float64
2,PREV_TRE_CN,3217556,55.507384,str
3,VOLCFNET,1952128,33.676964,float64
4,ACTUALHT,1807523,31.182324,float64
5,HTCD,1706518,29.439845,str
6,CR,1686578,29.095852,float64
7,FLDTYPCD,1329728,22.939685,str
8,CCLCD,1270533,21.918488,float64
9,BALIVE,1155702,19.937491,float64


In [8]:
initial_missing_summary[
    initial_missing_summary["variable"].isin(existing_key_columns)
].sort_values("missing_percentage", ascending=False)

,variable,missing_count,missing_percentage,dtype
4,ACTUALHT,1807523,31.182324,float64
6,CR,1686578,29.095852,float64
10,HT,1057730,18.247336,float64
15,CARBON_AG,738923,12.747465,float64
17,DIA,291608,5.030650,float64
22,STDAGE,127280,2.195760,float64
23,STDSZCD,97524,1.682427,float64
24,FORTYPCD,87003,1.500925,float64
25,SITECLCD,86865,1.498544,float64
26,OWNGRPCD,47515,0.819701,float64


## 6. Target variable validation

In [9]:
target_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "missing_count",
        "missing_percentage",
        "negative_count",
        "zero_count",
        "min",
        "median",
        "mean",
        "max"
    ],
    "value": [
        len(merged_df),
        merged_df[target_col].isna().sum(),
        merged_df[target_col].isna().mean() * 100,
        (merged_df[target_col] < 0).sum(),
        (merged_df[target_col] == 0).sum(),
        merged_df[target_col].min(),
        merged_df[target_col].median(),
        merged_df[target_col].mean(),
        merged_df[target_col].max()
    ]
})

target_summary

,metric,value
0,row_count,5.796627e+06
1,missing_count,7.389230e+05
2,missing_percentage,1.274747e+01
3,negative_count,0.000000e+00
4,zero_count,1.588150e+05
5,min,0.000000e+00
6,median,9.036593e+01
7,mean,4.813652e+02
8,max,1.804221e+05


In [10]:
target_summary.to_csv(OUTPUTS_DIR / "target_summary.csv", index=False)

In [11]:
# Convert key columns to numeric where appropriate
numeric_conversion_cols = [
    "CARBON_AG",
    "DIA",
    "HT",
    "ACTUALHT",
    "CR",
    "STDAGE",
    "BALIVE",
    "ALSTK",
    "GSSTK"
]

for col in numeric_conversion_cols:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

# Check converted data types
merged_df[[c for c in numeric_conversion_cols if c in merged_df.columns]].dtypes

CARBON_AG    float64
DIA          float64
HT           float64
ACTUALHT     float64
CR           float64
STDAGE       float64
BALIVE       float64
ALSTK        float64
GSSTK        float64
dtype: object

### Data type conversion note

Some FIA numeric fields were loaded as text because large CSV files can contain mixed or ambiguous types. Before applying biological validity checks, key numeric variables such as `CARBON_AG`, `DIA`, `HT`, `ACTUALHT`, `CR` and `STDAGE` were converted to numeric values using coercion. Invalid text values are converted to missing values and handled during the cleaning process.

## 7. Biological validity checks before cleaning

In [12]:
validity_checks = {}

if "DIA" in merged_df.columns:
    validity_checks["missing_DIA"] = merged_df["DIA"].isna().sum()
    validity_checks["DIA_less_equal_zero"] = (merged_df["DIA"] <= 0).sum()

if "HT" in merged_df.columns:
    validity_checks["missing_HT"] = merged_df["HT"].isna().sum()
    validity_checks["HT_less_equal_zero"] = (merged_df["HT"] <= 0).sum()

if "ACTUALHT" in merged_df.columns:
    validity_checks["missing_ACTUALHT"] = merged_df["ACTUALHT"].isna().sum()
    validity_checks["ACTUALHT_less_equal_zero"] = (merged_df["ACTUALHT"] <= 0).sum()

if "CR" in merged_df.columns:
    validity_checks["missing_CR"] = merged_df["CR"].isna().sum()
    validity_checks["CR_less_zero"] = (merged_df["CR"] < 0).sum()
    validity_checks["CR_greater_100"] = (merged_df["CR"] > 100).sum()

validity_df = pd.DataFrame(
    list(validity_checks.items()),
    columns=["check", "count"]
)

validity_df

,check,count
0,missing_DIA,291608
1,DIA_less_equal_zero,0
2,missing_HT,1057730
3,HT_less_equal_zero,1
4,missing_ACTUALHT,1807523
5,ACTUALHT_less_equal_zero,2
6,missing_CR,1686578
7,CR_less_zero,0
8,CR_greater_100,0


## 8. Tree status code distribution

In [13]:
if "STATUSCD" in merged_df.columns:
    status_counts = merged_df["STATUSCD"].value_counts(dropna=False).sort_index()
    display(status_counts)
else:
    print("STATUSCD column not found.")

STATUSCD
0     116563
1    4659376
2     923820
3      96868
Name: count, dtype: int64

For this project, the modelling target is above-ground carbon storage for standing trees. Therefore, the cleaning pipeline keeps live trees only using `STATUSCD == 1`, while documenting the number of records removed.

## 9. Apply cleaning rules

In [14]:
cleaned_df, cleaning_summary = clean_fia_tree_data(
    merged_df,
    target_col=target_col,
    keep_live_only=True
)

cleaning_summary

,step,rows_before,rows_after,rows_removed,percentage_removed,reason
0,Start,5796627,5796627,0,0.000000,Start with merged dataset.
1,Remove missing target,5796627,5057704,738923,12.747465,Remove rows where CARBON_AG is missing.
2,Remove negative target,5057704,5057704,0,0.000000,Remove rows where CARBON_AG is negative.
3,Remove missing DIA,5057704,5057704,0,0.000000,Remove rows where tree diameter is missing.
4,Remove DIA <= 0,5057704,5057704,0,0.000000,Remove rows where tree diameter is zero or neg...
5,Remove missing SPCD,5057704,5057704,0,0.000000,Remove rows where species code is missing.
6,Keep live trees only,5057704,4540549,517155,10.225094,Keep only rows where STATUSCD equals 1.
7,Remove exact duplicate rows,4540549,4540549,0,0.000000,Remove rows that are exact duplicates across a...


## 10. Validate cleaned dataset

In [15]:
print("Original merged shape:", merged_df.shape)
print("Cleaned shape:", cleaned_df.shape)
print("Rows removed:", len(merged_df) - len(cleaned_df))
print("Percentage removed:", round((len(merged_df) - len(cleaned_df)) / len(merged_df) * 100, 2))

Original merged shape: (5796627, 55)
Cleaned shape: (4540549, 55)
Rows removed: 1256078
Percentage removed: 21.67


In [16]:
post_clean_checks = {
    "missing_target": cleaned_df[target_col].isna().sum(),
    "negative_target": (cleaned_df[target_col] < 0).sum(),
    "missing_DIA": cleaned_df["DIA"].isna().sum(),
    "DIA_less_equal_zero": (cleaned_df["DIA"] <= 0).sum(),
    "missing_SPCD": cleaned_df["SPCD"].isna().sum(),
}

if "STATUSCD" in cleaned_df.columns:
    post_clean_checks["non_live_status_count"] = (cleaned_df["STATUSCD"] != 1).sum()

pd.DataFrame(list(post_clean_checks.items()), columns=["check", "count"])

,check,count
0,missing_target,0
1,negative_target,0
2,missing_DIA,0
3,DIA_less_equal_zero,0
4,missing_SPCD,0
5,non_live_status_count,0


## 11. Missing value summary after cleaning

In [17]:
post_missing_summary = summarise_missing_values(cleaned_df)

post_missing_summary.head(30)

,variable,missing_count,missing_percentage,dtype
0,VOLBFNET,2768722,60.977692,float64
1,VOLCSNET,2768722,60.977692,float64
2,PREV_TRE_CN,2722831,59.967000,str
3,ACTUALHT,1193992,26.296203,float64
4,HTCD,1170052,25.768954,str
5,FLDTYPCD,1070801,23.583073,str
6,VOLCFNET,1063705,23.426793,float64
7,BALIVE,942054,20.747579,float64
8,MEASDAY,852275,18.770307,float64
9,HT,522439,11.506076,float64


In [18]:
post_missing_summary[
    post_missing_summary["variable"].isin(existing_key_columns)
].sort_values("missing_percentage", ascending=False)

,variable,missing_count,missing_percentage,dtype
3,ACTUALHT,1193992,26.296203,float64
9,HT,522439,11.506076,float64
10,CR,431779,9.509401,float64
17,STDAGE,42054,0.926188,float64
18,STDSZCD,17204,0.378897,float64
19,SITECLCD,6929,0.152603,float64
20,FORTYPCD,6929,0.152603,float64
21,OWNGRPCD,2963,0.065256,float64
27,STATUSCD,0,0.000000,int64
28,SPCD,0,0.000000,int64


## 12. Key variable summary after cleaning

In [19]:
key_variable_summary = summarise_key_variables(cleaned_df, target_col=target_col)

key_variable_summary

,variable,summary_type,count,missing_percentage,mean,median,std,min,max,n_unique,top_value,top_value_frequency
0,CARBON_AG,numeric,4540549,0.000000,494.800823,96.494999,1535.001146,0.006032,180422.108817,None,None,None
1,DIA,numeric,4540549,0.000000,8.909114,7.000000,7.801642,1.000000,192.000000,None,None,None
2,HT,numeric,4018110,11.506076,52.090943,47.000000,30.882491,1.000000,331.000000,None,None,None
3,ACTUALHT,numeric,3346557,26.296203,53.246982,48.000000,31.122127,1.000000,331.000000,None,None,None
4,CR,numeric,4108770,9.509401,42.699439,40.000000,18.245625,0.000000,100.000000,None,None,None
5,SPCD,numeric,4540549,0.000000,332.983556,202.000000,292.602022,11.000000,8944.000000,None,None,None
6,STATECD,numeric,4540549,0.000000,19.494896,13.000000,16.105778,1.000000,53.000000,None,None,None
7,FORTYPCD,numeric,4533620,0.152603,373.894972,270.000000,248.684497,101.000000,999.000000,None,None,None
8,OWNGRPCD,numeric,4537586,0.065256,32.493945,40.000000,12.419025,10.000000,40.000000,None,None,None
9,SITECLCD,numeric,4533620,0.152603,4.424368,5.000000,1.134251,1.000000,7.000000,None,None,None


In [20]:
height_cr_cols = [c for c in ["HT", "ACTUALHT", "CR"] if c in cleaned_df.columns]

cleaned_df[height_cr_cols].isna().mean().mul(100).round(2)

HT          11.51
ACTUALHT    26.30
CR           9.51
dtype: float64

The missingness of `HT`, `ACTUALHT` and `CR` will be considered during later preprocessing. These variables are not removed automatically at this stage because they may still provide useful predictive information, and different modelling approaches can handle or impute missing values in different ways.

## 13. Save cleaned dataset and cleaning outputs

In [21]:
save_cleaning_outputs(
    cleaned_df=cleaned_df,
    cleaning_summary=cleaning_summary,
    missing_summary=post_missing_summary,
    key_summary=key_variable_summary
)

print("Saved cleaned dataset to:", INTERIM_DIR / "fia_tree_carbon_clean_unencoded.parquet")
print("Saved cleaning summary to:", OUTPUTS_DIR / "cleaning_summary.csv")
print("Saved missing value summary to:", OUTPUTS_DIR / "missing_value_summary.csv")
print("Saved key variable summary to:", OUTPUTS_DIR / "key_variable_summary.csv")

Cleaned dataset saved to E:\BA_DV_Project\COMM074_FIA_Project\data\interim\fia_tree_carbon_clean_unencoded.parquet
Cleaning summary saved to E:\BA_DV_Project\COMM074_FIA_Project\outputs\cleaning_summary.csv
Missing value summary saved to E:\BA_DV_Project\COMM074_FIA_Project\outputs\missing_value_summary.csv
Key variable summary saved to E:\BA_DV_Project\COMM074_FIA_Project\outputs\key_variable_summary.csv
Saved cleaned dataset to: E:\BA_DV_Project\COMM074_FIA_Project\data\interim\fia_tree_carbon_clean_unencoded.parquet
Saved cleaning summary to: E:\BA_DV_Project\COMM074_FIA_Project\outputs\cleaning_summary.csv
Saved missing value summary to: E:\BA_DV_Project\COMM074_FIA_Project\outputs\missing_value_summary.csv
Saved key variable summary to: E:\BA_DV_Project\COMM074_FIA_Project\outputs\key_variable_summary.csv


## 14. Verify saved cleaned dataset

In [22]:
reloaded_cleaned = pd.read_parquet(INTERIM_DIR / "fia_tree_carbon_clean_unencoded.parquet")

print("Reloaded shape:", reloaded_cleaned.shape)
print("Matches cleaned_df shape:", reloaded_cleaned.shape == cleaned_df.shape)

Reloaded shape: (4540549, 55)
Matches cleaned_df shape: True


## 15. Cleaning interpretation for the report

The merged tree-level dataset was cleaned using rules linked to the project objective of predicting individual-tree above-ground carbon storage. Records with missing or invalid target values were removed because supervised regression requires a known response variable. Records with missing or non-positive diameter were removed because diameter is a core biological measurement for individual trees and is expected to be a major predictor of carbon storage. The dataset was also restricted to live trees using `STATUSCD == 1` to align with the business focus on standing tree carbon storage.

Extreme values were not automatically removed at this stage because large trees may represent important high-carbon observations rather than errors. Outlier treatment will therefore be considered during EDA and final preprocessing, rather than applied blindly during initial cleaning.

## 16. Summary

This notebook cleaned the merged FIA tree-level dataset by removing records that could not support the supervised carbon storage prediction task. The cleaning process removed missing or invalid target values, missing or invalid diameter values, missing species codes, non-live tree records and exact duplicates.

The cleaned unencoded dataset has been saved as:

- `data/interim/fia_tree_carbon_clean_unencoded.parquet`

The following report-ready outputs were also saved:

- `outputs/cleaning_summary.csv`
- `outputs/missing_value_summary.csv`
- `outputs/key_variable_summary.csv`
- `outputs/target_summary.csv`
- `outputs/biological_validity_checks_before_cleaning.csv`

The next notebook will perform EDA on the cleaned dataset.